# Silver Layer - Data Cleaning & Integration

### Load bronze layer

In [42]:
import re
from pyspark.sql import DataFrame
from pyspark.sql import functions as F   # capital F

SRC = {
    "customers": "ShoppingMart_Bronze_Customers",
    "products":  "ShoppingMart_Bronze_Products",
    "orders":    "ShoppingMart_Bronze_Orders",
    "reviews":   "ShoppingMart_Bronze_Reviews",
    "social":    "ShoppingMart_Bronze_Social_Media",
    "web":       "ShoppingMart_Bronze_Web_Logs",
}

def snake_case_columns(df: DataFrame) -> DataFrame:
    """Standardise column names: trim, replace odd characters, lower-case."""
    for c in df.columns:
        new = re.sub(r"[^0-9a-zA-Z]+", "_", c.strip()).strip("_").lower()
        df = df.withColumnRenamed(c, new)
    return df

def read_csv_as_text(folder: str) -> DataFrame:
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)   # everything as string; we cast on purpose
            .csv(f"Files/{folder}"))

def read_json_lines(folder: str) -> DataFrame:
    return spark.read.json(f"Files/{folder}")   # JSON Lines: one object per line

def log(name: str, before: int, after: int) -> None:
    print(f"{name:<18} in: {before:>6}  out: {after:>6}  removed: {before - after:>5}")

StatementMeta(, d3a38777-a447-420b-b067-74c949b1c9ff, 44, Finished, Available, Finished, False)

### Explore data before cleaning

In [43]:
#Explore data before cleaning
"""
raw_orders = read_csv_as_text(SRC["orders"])
raw_orders.printSchema()
display(raw_orders.limit(10))
print(raw_orders.count())

"""

StatementMeta(, d3a38777-a447-420b-b067-74c949b1c9ff, 45, Finished, Available, Finished, False)

'\nraw_orders = read_csv_as_text(SRC["orders"])\nraw_orders.printSchema()\ndisplay(raw_orders.limit(10))\nprint(raw_orders.count())\n\n'

### Clean Data (trim,lower,to_date)

In [44]:
#Data Cleaning

customers_raw = snake_case_columns(read_csv_as_text(SRC["customers"]))
customers = (customers_raw
    .withColumn("customerid",   F.trim("customerid"))
    .withColumn("customername", F.initcap(F.trim("customername")))
    .withColumn("email",         F.lower(F.trim("email")))
    .withColumn("location",      F.trim("location"))
    .withColumn("signupdate",   F.to_date("signupdate"))
    .dropna(subset=["customerid"])
    .dropDuplicates(["customerid"]))

products_raw = snake_case_columns(read_csv_as_text(SRC["products"]))
products = (products_raw
    .withColumn("productid",     F.trim("productid"))
    .withColumn("productname",   F.trim("productname"))
    .withColumn("category",      F.initcap(F.trim("category")))
    .withColumn("stock",         F.col("stock").cast("int"))
    .withColumn("unitprice",     F.col("unitprice").cast("decimal(10,2)"))
    .dropna(subset=["productid", "unitprice"])
    .filter(F.col("unitprice") >= 0)
    .dropDuplicates(["productid"]))

orders_raw = snake_case_columns(read_csv_as_text(SRC["orders"]))
orders = (orders_raw
    .withColumn("orderid",       F.trim("orderid"))
    .withColumn("customerid",    F.trim("customerid"))
    .withColumn("productid",     F.trim("productid"))
    .withColumn("orderdate",     F.to_date("orderdate"))
    .withColumn("quantity",  F.col("quantity").cast("int"))
    .withColumn("totalamount",   F.col("totalamount").cast("decimal(12,2)"))
    .withColumn("paymentmethod", F.trim("paymentmethod"))
    .dropna(subset=["orderid", "customerid", "productid",
                    "orderdate", "totalamount"])
    .filter((F.length("customerid") > 0)
            & (F.col("quantity") > 0)
            & (F.col("totalamount") >= 0))
    .dropDuplicates(["orderid"]))

log("customers", customers_raw.count(), customers.count())
log("products",  products_raw.count(),  products.count())
log("orders",    orders_raw.count(),    orders.count())


StatementMeta(, d3a38777-a447-420b-b067-74c949b1c9ff, 46, Finished, Available, Finished, False)

customers          in:    100  out:    100  removed:     0
products           in:     50  out:     50  removed:     0
orders             in:   3000  out:   3000  removed:     0


### Validate & Deduplicate

In [45]:
# validate, deduplicate

valid_orders = (orders
    .join(customers.select("customerid"), "customerid", "left_semi")
    .join(products.select("productid"),   "productid",  "left_semi"))

# Raw rows (original text) that did not make it into Silver
orders_rejected = (orders_raw
    .withColumn("order_id_key", F.trim("orderid"))
    .join(valid_orders.select(F.col("orderid").alias("order_id_key")),
          "order_id_key", "left_anti")
    .drop("order_id_key")
    .dropDuplicates()
    .withColumn("rejected_at", F.current_timestamp()))

orders = valid_orders
log("orders", orders_raw.count(), orders.count())
print("rejected rows kept for review:", orders_rejected.count())

StatementMeta(, d3a38777-a447-420b-b067-74c949b1c9ff, 47, Finished, Available, Finished, False)

orders             in:   3000  out:   2939  removed:    61
rejected rows kept for review: 61


### snake_case with underscores

### Cleaning semi-structured data

In [46]:
#timestamp is a long, so it’s an epoch number, not a date string. 
def epoch_to_ts(col_name: str):                        
    """Epoch seconds or milliseconds -> timestamp."""
    c = F.col(col_name).cast("double")
    secs = F.when(c > 1e11, c / 1000.0).otherwise(c)
    return F.to_timestamp(F.from_unixtime(secs.cast("long")))

# Reviews
reviews_raw = snake_case_columns(read_json_lines(SRC["reviews"]))
reviews = (reviews_raw
    .withColumn("customer_id", F.trim("customer_id"))
    .withColumn("product_id",  F.trim("product_id"))
    .withColumn("rating",      F.col("rating").cast("int"))
    .withColumn("review_text", F.trim("review_text"))
    .withColumn("event_ts",    epoch_to_ts("timestamp"))
    .drop("timestamp")
    .dropna(subset=["product_id", "rating", "event_ts"])
    .filter(F.col("rating").between(1, 5))
    # optional: drop reviews for products not in Silver (products uses "productid")
    .join(products.select(F.col("productid").alias("product_id")),
          "product_id", "left_semi"))
log("product_reviews", reviews_raw.count(), reviews.count())

# Social media
social_raw = snake_case_columns(read_json_lines(SRC["social"]))
social = (social_raw
    .withColumn("content",   F.trim("content"))
    .withColumn("platform",  F.initcap(F.trim("platform")))
    .withColumn("sentiment", F.lower(F.trim("sentiment")))
    .withColumn("event_ts",  epoch_to_ts("timestamp"))
    .drop("timestamp")
    .dropna(subset=["platform", "sentiment", "event_ts"])
    .filter(F.col("sentiment").isin("positive", "neutral", "negative")))
log("social_media", social_raw.count(), social.count())

# Web logs
web_raw = snake_case_columns(read_json_lines(SRC["web"]))
web_logs = (web_raw
    .withColumn("user_id",  F.trim("user_id"))
    .withColumn("page",     F.lower(F.trim("page")))
    .withColumn("action",   F.lower(F.trim("action")))
    .withColumn("event_ts", epoch_to_ts("timestamp"))
    .drop("timestamp")
    .dropna(subset=["user_id", "page", "action", "event_ts"]))
log("web_logs", web_raw.count(), web_logs.count())

StatementMeta(, d3a38777-a447-420b-b067-74c949b1c9ff, 48, Finished, Available, Finished, False)

product_reviews    in:   3000  out:   2938  removed:    62
social_media       in:   3000  out:   3000  removed:     0
web_logs           in:   3000  out:   3000  removed:     0


### Save and verify

In [47]:
def save_table(df: DataFrame, name: str) -> None:
    (df.write
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .format("delta")
       .saveAsTable(name))
    print(f"saved {name}: {spark.table(name).count()} rows")

for df, name in [(customers,       "customers"),
                 (products,        "products"),
                 (orders,          "orders"),
                 (orders_rejected, "orders_rejected"),
                 (reviews,         "product_reviews"),
                 (social,          "social_media"),
                 (web_logs,        "web_logs")]:
    save_table(df, name)

spark.sql("SHOW TABLES").show()
spark.sql("DESCRIBE DETAIL orders").select("format", "numFiles", "sizeInBytes").show()
spark.table("orders").printSchema()
display(spark.sql("SELECT paymentmethod, COUNT(*) AS n FROM orders GROUP BY paymentmethod"))

StatementMeta(, d3a38777-a447-420b-b067-74c949b1c9ff, 49, Finished, Available, Finished, False)

saved customers: 100 rows
saved products: 50 rows
saved orders: 2939 rows
saved orders_rejected: 61 rows
saved product_reviews: 2938 rows
saved social_media: 3000 rows
saved web_logs: 3000 rows
+--------------------+---------------+-----------+
|           namespace|      tableName|isTemporary|
+--------------------+---------------+-----------+
|`E-comm_Workspace...|      customers|      false|
|`E-comm_Workspace...|         orders|      false|
|`E-comm_Workspace...|orders_rejected|      false|
|`E-comm_Workspace...|product_reviews|      false|
|`E-comm_Workspace...|       products|      false|
|`E-comm_Workspace...|   social_media|      false|
|`E-comm_Workspace...|       web_logs|      false|
+--------------------+---------------+-----------+

+------+--------+-----------+
|format|numFiles|sizeInBytes|
+------+--------+-----------+
| delta|       1|      53841|
+------+--------+-----------+

root
 |-- productid: string (nullable = true)
 |-- customerid: string (nullable = true)
 |-- 

SynapseWidget(Synapse.DataFrame, 1e3ad282-1760-4798-8371-5cfb01662f67)